# Perturbed Optimizer — Sigma Sweep Diagnostics

Configure the cell below, then run all cells.

In [ ]:

# ---- Configuration ----
PROBLEM   = "cubic"
VERSION   = "gen"
PREFIX    = "diag_run1"
RESULTS_DIR = f"../saved_records/{PROBLEM}-{VERSION}/perturb_sigma_sweep/{PREFIX}"


In [ ]:

import os
import glob
import re

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ---- Load all runs ----
files = glob.glob(os.path.join(RESULTS_DIR, "sigma_*.npz"))
assert len(files) > 0, f"No .npz files found in {RESULTS_DIR}"

runs = {}  # (sigma_float, model_type) -> dict of arrays
for f in sorted(files):
    fname = os.path.basename(f)
    m = re.match(r"sigma_(.+)_(\w+)\.npz", fname)
    if not m:
        continue
    sigma, model_type = float(m.group(1)), m.group(2)
    runs[(sigma, model_type)] = dict(np.load(f))

model_types = sorted({mt for _, mt in runs})
sigma_values = sorted({s for s, _ in runs})

print(f"Loaded {len(runs)} runs")
print(f"  model types : {model_types}")
print(f"  sigma values: {[f'{s:.4g}' for s in sigma_values]}")


In [ ]:

# ---- Colour palette: one colour per sigma, log-mapped ----
log_sigmas = np.log10(sigma_values)
norm = plt.Normalize(log_sigmas.min(), log_sigmas.max())
cmap = cm.plasma

def sigma_color(s):
    return cmap(norm(np.log10(s)))

def add_colorbar(ax, label="σ"):
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label(label)
    n_ticks = min(len(sigma_values), 6)
    tick_idx = np.linspace(0, len(sigma_values)-1, n_ticks, dtype=int)
    cbar.set_ticks([log_sigmas[i] for i in tick_idx])
    cbar.set_ticklabels([f"{sigma_values[i]:.3g}" for i in tick_idx])


## Training curves — val regret and loss

In [ ]:

fig, axes = plt.subplots(2, len(model_types), figsize=(7*len(model_types), 8), sharey="row")
if len(model_types) == 1:
    axes = axes[:, np.newaxis]

for col, mt in enumerate(model_types):
    for sigma in sigma_values:
        key = (sigma, mt)
        if key not in runs:
            continue
        d = runs[key]
        epochs = d["epoch"]
        c = sigma_color(sigma)
        axes[0, col].plot(epochs, d["val_regret"],   color=c, lw=1.5)
        axes[1, col].plot(epochs, d["loss"],          color=c, lw=1.5)

    axes[0, col].set_title(f"Val regret — {mt}", fontsize=13)
    axes[1, col].set_title(f"Training loss — {mt}", fontsize=13)
    for row in range(2):
        axes[row, col].set_xlabel("Epoch")
        add_colorbar(axes[row, col])

axes[0, 0].set_ylabel("Normalised regret")
axes[1, 0].set_ylabel("Perturbed loss")
fig.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_training_curves.png"), dpi=150)
plt.show()


## Gradient quality — cosine similarity with FD gradient

In [ ]:

fig, axes = plt.subplots(1, len(model_types), figsize=(7*len(model_types), 4), sharey=True)
if len(model_types) == 1:
    axes = [axes]

for ax, mt in zip(axes, model_types):
    for sigma in sigma_values:
        key = (sigma, mt)
        if key not in runs:
            continue
        d = runs[key]
        cos = d["cosine_sim"]
        valid = ~np.isnan(cos)
        ax.plot(d["epoch"][valid], cos[valid], color=sigma_color(sigma), lw=1.5)

    ax.axhline(0, color="k", lw=0.7, ls="--")
    ax.set_title(f"Cosine sim (perturb grad vs FD grad) — {mt}", fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylim(-1.05, 1.05)
    add_colorbar(ax)

axes[0].set_ylabel("Cosine similarity")
fig.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_cosine_sim.png"), dpi=150)
plt.show()


## Rank change rate and interiority over training

In [ ]:

metrics = [
    ("rank_change_rate", "Rank change rate",   (0, 1)),
    ("softness",         "Softness z̄(1−z̄)",   (0, 0.26)),
    ("dist_binary",      "Dist to binary",      None),
    ("entropy",          "Entropy H(z̄)",        None),
]

fig, axes = plt.subplots(len(metrics), len(model_types),
                         figsize=(7*len(model_types), 4*len(metrics)), sharey="row")
if len(model_types) == 1:
    axes = axes[:, np.newaxis]

for row, (key, label, ylim) in enumerate(metrics):
    for col, mt in enumerate(model_types):
        ax = axes[row, col]
        for sigma in sigma_values:
            d = runs.get((sigma, mt))
            if d is None:
                continue
            vals = d[key]
            valid = ~np.isnan(vals)
            ax.plot(d["epoch"][valid], vals[valid], color=sigma_color(sigma), lw=1.5)
        ax.set_title(f"{label} — {mt}", fontsize=12)
        ax.set_xlabel("Epoch")
        if ylim:
            ax.set_ylim(*ylim)
        add_colorbar(ax)
    axes[row, 0].set_ylabel(label)

fig.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_interiority.png"), dpi=150)
plt.show()


## σ / ‖ĉ‖ ratio over training

In [ ]:

fig, axes = plt.subplots(1, len(model_types), figsize=(7*len(model_types), 4), sharey=True)
if len(model_types) == 1:
    axes = [axes]

for ax, mt in zip(axes, model_types):
    for sigma in sigma_values:
        d = runs.get((sigma, mt))
        if d is None:
            continue
        ax.plot(d["epoch"], d["sigma_ratio"], color=sigma_color(sigma), lw=1.5)

    ax.axhline(1.0, color="k", lw=0.7, ls="--", label="ratio = 1")
    ax.set_yscale("log")
    ax.set_title(f"σ / ‖ĉ‖ — {mt}", fontsize=12)
    ax.set_xlabel("Epoch")
    add_colorbar(ax)

axes[0].set_ylabel("σ / mean|coeff_hat|  (log scale)")
fig.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_sigma_ratio.png"), dpi=150)
plt.show()


## Key scatter — cosine similarity vs rank change rate

Each point is one epoch from one run. Color = sigma. The hypothesis: there is a sweet-spot rank_change_rate where gradient quality (cosine sim) is maximised.

In [ ]:

fig, axes = plt.subplots(1, len(model_types), figsize=(7*len(model_types), 5), sharey=True)
if len(model_types) == 1:
    axes = [axes]

for ax, mt in zip(axes, model_types):
    for sigma in sigma_values:
        d = runs.get((sigma, mt))
        if d is None:
            continue
        rcr = d["rank_change_rate"]
        cos = d["cosine_sim"]
        valid = ~np.isnan(rcr) & ~np.isnan(cos)
        ax.scatter(rcr[valid], cos[valid],
                   color=sigma_color(sigma), alpha=0.5, s=12, label=f"σ={sigma:.3g}")

    ax.axhline(0, color="k", lw=0.7, ls="--")
    ax.set_xlim(0, 1)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("Rank change rate")
    ax.set_title(f"Gradient quality vs exploration — {mt}", fontsize=12)
    add_colorbar(ax)

axes[0].set_ylabel("Cosine similarity (perturb grad vs FD grad)")
fig.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig_scatter_cos_vs_rcr.png"), dpi=150)
plt.show()


## Final-epoch summary table

In [ ]:

import pandas as pd

rows = []
for (sigma, mt), d in sorted(runs.items()):
    rows.append({
        "sigma":           sigma,
        "model":           mt,
        "val_regret":      d["val_regret"][-1],
        "cosine_sim":      d["cosine_sim"][~np.isnan(d["cosine_sim"])][-1] if (~np.isnan(d["cosine_sim"])).any() else np.nan,
        "rank_chg_rate":   d["rank_change_rate"][-1],
        "softness":        d["softness"][-1],
        "sigma_ratio":     d["sigma_ratio"][-1],
    })

df = pd.DataFrame(rows).sort_values(["model", "sigma"])
df = df.set_index(["model", "sigma"])
pd.set_option("display.float_format", "{:.4f}".format)
df
